In [ ]:

import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from tqdm import tqdm
import re


random.seed(42)
torch.manual_seed(42)
np.random.seed(42)


print("Loading GSM8K dataset...")
gsm8k = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]
eval_data = gsm8k["test"]

indices = np.load("/projects/data/kundeshwar_pundalik/curious/selected_samples_1200.npy") 
subset = train_data.select(set(indices.tolist()))


model_name = "microsoft/phi-3-mini-4k-instruct"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


def preprocess_function(examples):
    inputs = []
    targets = []
    for question, answer in zip(examples["question"], examples["answer"]):
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs.append(prompt)
        targets.append(answer)
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"].copy()
    for i in range(len(model_inputs["labels"])):
        model_inputs["labels"][i] = [label if label != tokenizer.pad_token_id else -100 for label in model_inputs["labels"][i]]
    return model_inputs

def extract_answer(text):
    numbers = re.findall(r'\d+', text)
    if numbers:
        return numbers[-1]
    return ""


def evaluate_gsm8k(model, tokenizer, eval_dataset, num_samples=100):
    if num_samples < len(eval_dataset):
        eval_subset = eval_dataset.select(range(num_samples))
    else:
        eval_subset = eval_dataset
    correct = 0
    total = 0
    model.eval()
    for example in tqdm(eval_subset, desc="Evaluating"):
        question = example["question"]
        gold_answer = example["answer"]
        gold_number = extract_answer(gold_answer)
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                num_beams=1,
            )
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = generated_text[len(prompt):]
        predicted_number = extract_answer(generated_text)
        if predicted_number == gold_number and gold_number != "":
            correct += 1
        total += 1
    accuracy = correct / total if total > 0 else 0
    return accuracy

# Fine-tune and evaluate
def finetune_and_evaluate(subset, eval_data):
    print(f"\n--- Fine-tuning with provided subset of size: {len(subset)} ---")
    print("Preprocessing data...")
    train_dataset = subset.map(
        preprocess_function,
        batched=True,
        remove_columns=subset.column_names,
        desc="Preprocessing training data"
    )
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    training_args = TrainingArguments(
        output_dir="./phi3_gsm8k_custom_subset",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        num_train_epochs=3,
        logging_steps=10,
        save_strategy="no",
        fp16=False,
        bf16=True,
        report_to="none",
        push_to_hub=False,
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_ratio=0.03,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        return_tensors="pt",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    print("Starting fine-tuning...")
    trainer.train()
    print("Evaluating model...")
    accuracy = evaluate_gsm8k(model, tokenizer, eval_data, num_samples=100)
    print(f"Subset size: {len(subset)} | Accuracy: {accuracy:.4f}")
    del model
    torch.cuda.empty_cache()
    return accuracy

# Run the process
accuracy = finetune_and_evaluate(subset, eval_data)
print(f"\nFinal test accuracy on GSM8K: {accuracy:.4f}")


Loading GSM8K dataset...
Loading tokenizer for microsoft/phi-3-mini-4k-instruct...

--- Fine-tuning with provided subset of size: 1200 ---
Preprocessing data...


Preprocessing training data:   0%|          | 0/1200 [00:00<?, ? examples/s]

Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_1540388/3160061551.py:128: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting fine-tuning...


Step,Training Loss
10,8.455000
20,5.842100
30,3.512100
40,2.365600
50,1.848300
60,1.730000
70,1.851100
80,1.671100
90,1.648900
100,1.644100


Evaluating model...


Evaluating:  30%|███       | 30/100 [02:31<04:21,  3.73s/it]

In [ ]:
 print(f"Subset size: {len(subset)})

Loading GSM8K dataset...
